# Encoder-only Zero-shot Batch Integration Benchmark

这个 Notebook 只做一件事：从 checkpoint 加载 Encoder 侧权重，冻结全部参数，提取 `cell_emb`，然后用统一指标评估 Batch Integration。

不包含训练、优化器、下游分类头、GRL、Early Stopping 或 Full Finetune。cell type 和 batch 标签只用于最终指标，不参与 embedding 生成。

Checkpoint 中只加载这些模块：`shared_token_embedding / token_norm / value_encoder / mask_flag_embedding / encoder`。Decoder、LM head 和额外 activity head 权重均不加载。模型前向仍调用原始模型类的 Encoder 路径，以保证输入组合、CLS pooling 和预训练时完全一致。


In [1]:
import gc
import hashlib
import importlib.util
import json
import os
import random
import traceback
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import scanpy as sc
except ImportError as e:
    raise ImportError('缺少 scanpy，请先安装 scanpy、igraph 和 leidenalg。') from e


/root/miniconda3/envs/cllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 唯一配置区

后续单元格不需要修改。比较多个同架构 checkpoint 时，只需在 `models` 中继续添加配置。


In [7]:
USER_CONFIG: Dict[str, Any] = {
    'pipeline_version': 'zeroshot-encoder-v1.1-native-51bin',
    'seed': 42,
    'interface_dir': '/root/autodl-tmp/Batch_integration',
    'dataset_name': 'Renal',
    'split_root': '/root/autodl-tmp/Batch_integration/data/Renal',
    'output_base_root': '/root/autodl-tmp/Batch_integration/outputs_zeroshot_sckode',
    'skip_finished': False,
    'evaluation_split': 'all',  # Zero-shot 无训练泄漏；论文主表建议 all。也可改为 test。
    'l2_normalize_for_evaluation': False,  # 与 scGPT CLS embedding 评估保持一致。
    'amp': True,
    'amp_dtype': 'auto',  # auto 优先 BF16，否则 FP16。
    'models': [
        {
            'model_name': 'geosketch_50_encoderonly',
            'model_py': '/root/autodl-tmp/ExpertCoder/ExpertCoder_Separate_decoder_activate_regulon/model.py',
            'checkpoint_path': '/root/autodl-tmp/pt/geosketch_50_encoderonly.pt',
            'global_vocab_size': 91714,
            'pad_token_id': 0,
            'd_model': 512,
            'n_heads': 8,
            'n_layers': 12,
            'decoder_n_layers': 2,
            'decoder_n_heads': 8,
            'expansion_ratio': 4,
            'dropout': 0.1,
            'mask_value': -3.0,
            'max_decoder_length': 512,
            'value_hidden_dim': 128,
            'value_head_hidden_dim': 256,
            'tie_lm_head': True,
            'use_mask_flag_embedding': True,
        }
    ],
    'data': {
        'max_length': 2049,
        'keep_first_n_tokens': 1,
        'num_bins': 51,
        'input_values_are_binned': False,  # Benchmark MDS 保存连续表达值；必须复现训练时逐细胞分箱。
        'cls_token_id': 1,
        'pad_token_id': 0,
        'cls_expr_value': -1.0,
        'pad_expr_value': -2.0,
        'batch_size': 18,
        'num_workers': 0,
    },
    'metrics': {
        'n_neighbors': 15,
        'leiden_resolution': 1.0,
        'min_cells_per_celltype': 30,
        'min_batches_per_celltype': 2,
        'silhouette_sample_size': None,
        'bio_weight': 0.6,
        'batch_weight': 0.4,
    },
}

GLOBAL_SEED = int(USER_CONFIG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

interface_dir = Path(USER_CONFIG['interface_dir'])
if str(interface_dir) not in __import__('sys').path:
    __import__('sys').path.append(str(interface_dir))
from batch_integration import (
    MDSBatchIntegrationConfig, inspect_one_batch, prepare_split_mds_for_batch_integration,
)

DATASET_NAME = str(USER_CONFIG['dataset_name'])
OUTPUT_ROOT = Path(USER_CONFIG['output_base_root']) / DATASET_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
data_cfg = USER_CONFIG['data']
MDS_CFG = MDSBatchIntegrationConfig(
    split_root=str(USER_CONFIG['split_root']),
    output_root=str(OUTPUT_ROOT), dataset_name=DATASET_NAME,
    train_dir_name='train', val_dir_name='val', test_dir_name='test',
    gene_key='genes', expr_key='expressions', cell_type_key='cell_type',
    batch_key='batch', cell_id_key='cell_id',
    max_length=int(data_cfg['max_length']), add_cls_token=True,
    cls_token_id=int(data_cfg['cls_token_id']), pad_token_id=int(data_cfg['pad_token_id']),
    cls_expr_value=float(data_cfg['cls_expr_value']), pad_expr_value=float(data_cfg['pad_expr_value']),
    batch_size_train=int(data_cfg['batch_size']), batch_size_eval=int(data_cfg['batch_size']),
    num_workers=int(data_cfg['num_workers']), pin_memory=True,
    seed=GLOBAL_SEED, save_metadata=True,
)
context = prepare_split_mds_for_batch_integration(MDS_CFG)
inspect_one_batch(context, split='train')

CFG: Dict[str, Any] = {
    'pipeline_version': USER_CONFIG['pipeline_version'],
    'dataset_name': DATASET_NAME, 'device': DEVICE,
    'skip_finished': bool(USER_CONFIG['skip_finished']),
    'evaluation_split': str(USER_CONFIG['evaluation_split']),
    'l2_normalize_for_evaluation': bool(USER_CONFIG['l2_normalize_for_evaluation']),
    'input_values_are_binned': bool(data_cfg['input_values_are_binned']),
    'num_bins': int(data_cfg['num_bins']),
    'keep_first_n_tokens': int(data_cfg['keep_first_n_tokens']),
    'cls_expr_value': float(data_cfg['cls_expr_value']),
    'pad_expr_value': float(data_cfg['pad_expr_value']),
    'amp': bool(USER_CONFIG['amp']), 'amp_dtype': USER_CONFIG['amp_dtype'],
    **USER_CONFIG['metrics'],
}
print('device:', DEVICE)
print('evaluation split:', CFG['evaluation_split'])
print('output:', OUTPUT_ROOT)


Scan metadata: test: 100%|██████████| 9713/9713 [00:00<00:00, 31269.16it/s]


MDS batch integration接口准备完成。
dataset: Renal
n_cells: 97125
n_train: 77700
n_val: 9712
n_test: 9713
n_cell_type: 35
n_batch: 3
genes: torch.Size([18, 2049]) torch.int64
expressions: torch.Size([18, 2049]) torch.float32
padding_mask: torch.Size([18, 2049]) torch.bool
cell_type_id: torch.Size([18]) torch.int64
batch_id: torch.Size([18]) torch.int64
row_idx: tensor([53488, 63376, 67961, 17608, 40654, 24790, 68142, 16219, 41523, 73264])
split: ['train', 'train', 'train', 'train', 'train']
device: cuda
evaluation split: all
output: /root/autodl-tmp/Batch_integration/outputs_zeroshot_sckode/Renal


## 2. 只加载 Encoder 侧权重并提取 cell embedding


In [8]:
ENCODER_MODULE_NAMES = (
    'shared_token_embedding', 'token_norm', 'value_encoder', 'mask_flag_embedding', 'encoder'
)
ENCODER_PREFIXES = tuple(f'{name}.' for name in ENCODER_MODULE_NAMES)


def resolve_amp_dtype(cfg: Dict[str, Any]):
    if not str(cfg['device']).startswith('cuda'):
        return torch.float32
    requested = str(cfg.get('amp_dtype', 'auto')).lower()
    if requested in {'auto', 'bf16', 'bfloat16'} and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def autocast_context(cfg: Dict[str, Any]):
    enabled = bool(cfg.get('amp', False)) and str(cfg['device']).startswith('cuda')
    if not enabled:
        return nullcontext()
    return torch.amp.autocast('cuda', dtype=resolve_amp_dtype(cfg), enabled=True)


def move_batch_to_device(batch: Dict[str, Any], device: str) -> Dict[str, Any]:
    return {
        key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def load_python_module_from_path(module_path: str, module_name: str):
    path = Path(module_path)
    if not path.exists():
        raise FileNotFoundError(f'模型文件不存在：{path}')
    spec = importlib.util.spec_from_file_location(module_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f'无法加载模型模块：{path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def clean_state_dict(checkpoint_path: str) -> Tuple[Dict[str, torch.Tensor], Dict[str, Any]]:
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    metadata = checkpoint if isinstance(checkpoint, dict) else {}
    state: Any = checkpoint
    if isinstance(checkpoint, dict):
        for key in ['model_state_dict', 'encoder_state_dict', 'state_dict', 'model']:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                state = checkpoint[key]
                print('checkpoint state source:', key)
                break
    if not isinstance(state, dict):
        raise TypeError(f'无法解析 checkpoint state：{type(state)}')
    clean: Dict[str, torch.Tensor] = {}
    for key, value in state.items():
        if not torch.is_tensor(value):
            continue
        name = str(key)
        changed = True
        while changed:
            changed = False
            for prefix in ['module.', '_orig_mod.', 'model.']:
                if name.startswith(prefix):
                    name = name[len(prefix):]
                    changed = True
        clean[name] = value
    return clean, metadata


def build_model_shell(model_cfg: Dict[str, Any]) -> nn.Module:
    module = load_python_module_from_path(model_cfg['model_py'], 'stage2_model_module_zeroshot')
    model_cls = getattr(module, 'TahoeStage2MixedModel', None)
    if model_cls is None:
        model_cls = getattr(module, 'TahoeStage2PromptModel')
    return model_cls(
        global_vocab_size=int(model_cfg['global_vocab_size']),
        d_model=int(model_cfg['d_model']), n_heads=int(model_cfg['n_heads']),
        n_layers=int(model_cfg['n_layers']),
        decoder_n_layers=int(model_cfg['decoder_n_layers']),
        decoder_n_heads=int(model_cfg['decoder_n_heads']),
        expansion_ratio=int(model_cfg['expansion_ratio']),
        dropout=float(model_cfg['dropout']),
        pad_token_id=int(model_cfg['pad_token_id']),
        mask_value=float(model_cfg['mask_value']),
        max_decoder_length=int(model_cfg['max_decoder_length']),
        value_hidden_dim=int(model_cfg['value_hidden_dim']),
        value_head_hidden_dim=int(model_cfg['value_head_hidden_dim']),
        tie_lm_head=bool(model_cfg['tie_lm_head']),
        use_mask_flag_embedding=bool(model_cfg['use_mask_flag_embedding']),
    )


def load_encoder_side_weights(model: nn.Module, checkpoint_path: str) -> Dict[str, Any]:
    state, metadata = clean_state_dict(checkpoint_path)
    model_state = model.state_dict()
    required = sorted(key for key in model_state if key.startswith(ENCODER_PREFIXES))
    available = sorted(key for key in state if key.startswith(ENCODER_PREFIXES))
    missing = [key for key in required if key not in state]
    unexpected = [key for key in available if key not in model_state]
    mismatch = [
        key for key in required
        if key in state and tuple(state[key].shape) != tuple(model_state[key].shape)
    ]
    if missing or unexpected or mismatch:
        raise RuntimeError(
            f'Encoder checkpoint 不匹配：missing={len(missing)}, '
            f'unexpected={len(unexpected)}, shape_mismatch={len(mismatch)}; '
            f'examples={missing[:5] + unexpected[:5] + mismatch[:5]}'
        )
    encoder_state = {key: state[key] for key in required}
    load_result = model.load_state_dict(encoder_state, strict=False)
    if load_result.unexpected_keys:
        raise RuntimeError(f'加载 Encoder 时出现意外键：{load_result.unexpected_keys[:20]}')
    ignored = [key for key in state if not key.startswith(ENCODER_PREFIXES)]
    print('encoder modules:', ENCODER_MODULE_NAMES)
    print(f'[OK] loaded encoder tensors={len(encoder_state)}; ignored non-encoder tensors={len(ignored)}')
    if ignored:
        print('ignored examples:', ignored[:12])
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.eval()
    return {
        'encoder_tensor_count': len(encoder_state),
        'ignored_tensor_count': len(ignored),
        'checkpoint_epoch': metadata.get('epoch'),
        'checkpoint_global_step': metadata.get('global_step'),
        'checkpoint_best_metric': metadata.get('best_metric'),
        'checkpoint_best_metric_name': metadata.get('best_metric_name'),
    }


def left_binning(values: torch.Tensor, num_bins: int) -> torch.Tensor:
    """与训练 data.py 一致：0 保持 0，正表达值按单细胞分位数映射到 1..num_bins-1。"""
    if values.numel() == 0:
        return values
    if not torch.isfinite(values).all():
        raise ValueError('表达值包含 NaN/Inf，不能执行分箱。')
    if values.lt(0).any():
        raise ValueError('基因表达区出现负值；CLS/PAD 必须在分箱前排除。')
    out = torch.zeros_like(values, dtype=torch.long)
    pos_mask = values.gt(0)
    if not pos_mask.any():
        return out
    pos_values = values[pos_mask].float()
    if pos_values.numel() == 1:
        out[pos_mask] = 1
        return out
    q = torch.linspace(0.0, 1.0, steps=int(num_bins), device=values.device)
    edges = torch.quantile(pos_values, q)
    inner_edges = edges[1:-1].contiguous()
    out[pos_mask] = torch.bucketize(pos_values, inner_edges, right=False).long() + 1
    return out


def prepare_encoder_values(batch: Dict[str, Any], cfg: Dict[str, Any]) -> torch.Tensor:
    values = batch['expressions'].float().clone()
    padding_mask = batch['padding_mask'].bool()
    keep_first = int(cfg['keep_first_n_tokens'])
    num_bins = int(cfg['num_bins'])
    if bool(cfg['input_values_are_binned']):
        valid_body = ~padding_mask
        valid_body[:, :keep_first] = False
        body = values[valid_body]
        if body.numel() and (body.lt(0).any() or body.gt(num_bins - 1).any() or not torch.allclose(body, body.round())):
            raise ValueError('配置声明输入已分箱，但有效基因值并非 0..num_bins-1 的整数。')
        return values

    # MDS 已添加 CLS 并在 sampling=False 下截断到 2049；随后逐细胞分箱。
    for row in range(values.shape[0]):
        valid = ~padding_mask[row]
        body = valid.clone()
        body[:keep_first] = False
        values[row, body] = left_binning(values[row, body], num_bins).to(values.dtype)
    return values


def audit_encoder_input(loader: DataLoader, cfg: Dict[str, Any]) -> Dict[str, Any]:
    batch = next(iter(loader))
    raw = batch['expressions'].float()
    padding_mask = batch['padding_mask'].bool()
    keep_first = int(cfg['keep_first_n_tokens'])
    valid_body = ~padding_mask
    valid_body[:, :keep_first] = False
    raw_body = raw[valid_body]
    processed = prepare_encoder_values(batch, cfg)
    processed_body = processed[valid_body]
    if raw_body.numel() == 0:
        raise RuntimeError('首个 batch 没有有效基因 token。')
    audit = {
        'input_values_are_binned': bool(cfg['input_values_are_binned']),
        'num_bins': int(cfg['num_bins']),
        'raw_min': float(raw_body.min().item()),
        'raw_max': float(raw_body.max().item()),
        'raw_non_integer_ratio': float((raw_body.sub(raw_body.round()).abs().gt(1e-6)).float().mean().item()),
        'binned_min': float(processed_body.min().item()),
        'binned_max': float(processed_body.max().item()),
        'binned_unique_preview': [float(x) for x in torch.unique(processed_body)[:60].tolist()],
        'valid_length_min': int((~padding_mask).sum(1).min().item()),
        'valid_length_max': int((~padding_mask).sum(1).max().item()),
        'cls_value_matches': bool(torch.allclose(raw[:, 0], torch.full_like(raw[:, 0], float(cfg['cls_expr_value'])))),
    }
    if audit['binned_min'] < 0 or audit['binned_max'] > int(cfg['num_bins']) - 1:
        raise RuntimeError(f"分箱结果越界：{audit}")
    if not audit['cls_value_matches']:
        raise RuntimeError(f"CLS expression value 与训练配置不一致：{audit}")
    print('native preprocessing audit:', json.dumps(audit, ensure_ascii=False))
    return audit


def encode_batch(model: nn.Module, batch: Dict[str, Any], device: str, cfg: Dict[str, Any]) -> torch.Tensor:
    # 与训练 collator 一致，在 CPU/FP32 完成分箱后再搬到 GPU。
    prepared_values = prepare_encoder_values(batch, cfg)
    batch = move_batch_to_device(batch, device)
    genes = batch['genes'].long()
    padding_mask = batch['padding_mask'].bool()
    values = prepared_values.to(device, non_blocking=True)
    encoder_outputs, _ = model.encode(
        encoder_input_gene_ids=genes,
        encoder_input_values=values,
        encoder_key_padding_mask=padding_mask,
        return_last_attn=False,
    )
    output = {'cell_emb': model.pool_cell_embedding(encoder_outputs, encoder_key_padding_mask=padding_mask)}
    if 'cell_emb' not in output:
        raise KeyError(f"模型输出缺少 cell_emb，现有键：{list(output.keys())}")
    embedding = output['cell_emb']
    if embedding.ndim != 2:
        raise ValueError(f'cell_emb 必须为 [B,H]，当前为 {tuple(embedding.shape)}')
    return embedding


def encode_all_cells(
    model: nn.Module, loader: DataLoader, device: str, cfg: Dict[str, Any], desc: str
) -> Tuple[np.ndarray, np.ndarray]:
    embeddings: List[np.ndarray] = []
    row_indices: List[np.ndarray] = []
    model.eval()
    with torch.inference_mode():
        for batch in tqdm(loader, desc=desc):
            with autocast_context(cfg):
                embedding = encode_batch(model, batch, device, cfg)
            embeddings.append(embedding.detach().cpu().float().numpy())
            row_indices.append(batch['row_idx'].detach().cpu().numpy().astype(np.int64))
    if not embeddings:
        raise RuntimeError('embed_loader_all 为空。')
    emb = np.concatenate(embeddings, axis=0).astype(np.float32)
    row_idx = np.concatenate(row_indices).astype(np.int64)
    if len(emb) != len(row_idx) or len(np.unique(row_idx)) != len(row_idx):
        raise ValueError('embedding 与 row_idx 不一致，或 row_idx 存在重复。')
    return emb, row_idx


## 3. 统一指标计算


In [9]:
def l2_normalize(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)


def select_evaluation_rows(
    emb: np.ndarray, row_idx: np.ndarray, context: Any, cfg: Dict[str, Any]
) -> Tuple[np.ndarray, np.ndarray]:
    split = str(cfg['evaluation_split']).lower()
    if split == 'all':
        return emb, row_idx
    if split != 'test':
        raise ValueError(f"evaluation_split 只支持 'all' 或 'test'，当前为 {split}")
    target_parts = [
        batch['row_idx'].detach().cpu().numpy().astype(np.int64)
        for batch in context.test_loader
    ]
    target = np.concatenate(target_parts)
    position = {int(row): i for i, row in enumerate(row_idx.tolist())}
    missing = [int(row) for row in target if int(row) not in position]
    if missing:
        raise KeyError(f'缺少 test row_idx：{missing[:20]}')
    take = np.asarray([position[int(row)] for row in target], dtype=np.int64)
    return emb[take], target


def safe_silhouette(
    x: np.ndarray, labels: np.ndarray, sample_size: Optional[int]
) -> float:
    labels = np.asarray(labels).astype(str)
    n_labels = len(np.unique(labels))
    if n_labels < 2 or len(labels) <= n_labels:
        return float('nan')
    try:
        kwargs: Dict[str, Any] = {'metric': 'euclidean'}
        if sample_size is not None and len(labels) > int(sample_size):
            kwargs.update(sample_size=int(sample_size), random_state=GLOBAL_SEED)
        return float(silhouette_score(np.asarray(x, np.float32), labels, **kwargs))
    except Exception as e:
        print('[WARNING] silhouette failed:', repr(e))
        return float('nan')


def global_knn_graph(emb: np.ndarray, n_neighbors: int):
    from scipy import sparse
    n_cells = int(len(emb))
    if n_cells <= 1:
        return sparse.csr_matrix((n_cells, n_cells), dtype=np.float32)
    k = min(int(n_neighbors), n_cells - 1)
    neighbors = NearestNeighbors(n_neighbors=k + 1, metric='euclidean')
    indices = neighbors.fit(emb).kneighbors(emb, return_distance=False)[:, 1:]
    rows = np.repeat(np.arange(n_cells), k)
    cols = indices.reshape(-1)
    graph = sparse.csr_matrix(
        (np.ones(len(rows), np.float32), (rows, cols)), shape=(n_cells, n_cells)
    )
    return graph.maximum(graph.T)


def conditioned_batch_metrics(
    emb: np.ndarray, cell_types: np.ndarray, batches: np.ndarray, cfg: Dict[str, Any]
) -> Tuple[float, float, pd.DataFrame]:
    graph = global_knn_graph(emb, int(cfg['n_neighbors']))
    rows: List[Dict[str, Any]] = []
    for cell_type in sorted(np.unique(cell_types.astype(str))):
        idx = np.where(cell_types.astype(str) == str(cell_type))[0]
        batch_sub = batches[idx].astype(str)
        if len(idx) < int(cfg['min_cells_per_celltype']):
            continue
        if len(np.unique(batch_sub)) < int(cfg['min_batches_per_celltype']):
            continue
        raw_batch = safe_silhouette(emb[idx], batch_sub, cfg['silhouette_sample_size'])
        asw_batch = float(np.clip(1.0 - abs(raw_batch), 0.0, 1.0))
        subgraph = graph[idx][:, idx]
        _, components = connected_components(subgraph, directed=False, connection='weak')
        graph_conn = float(np.bincount(components).max() / len(idx))
        rows.append({
            'cell_type': str(cell_type), 'n_cells': int(len(idx)),
            'n_batches': int(len(np.unique(batch_sub))),
            'ASW_batch': asw_batch, 'GraphConn': graph_conn,
        })
    detail = pd.DataFrame(rows)
    if detail.empty:
        return float('nan'), float('nan'), detail
    return float(detail['ASW_batch'].mean()), float(detail['GraphConn'].mean()), detail


def evaluate_embedding(
    emb: np.ndarray, row_idx: np.ndarray, context: Any, run_dir: Path, cfg: Dict[str, Any]
) -> Dict[str, Any]:
    cell_ids = np.asarray(context.cell_type_ids)[row_idx]
    batch_ids = np.asarray(context.batch_ids)[row_idx]
    cell_types = np.asarray([context.id_to_cell_type[int(x)] for x in cell_ids]).astype(str)
    batches = np.asarray([context.id_to_batch[int(x)] for x in batch_ids]).astype(str)
    adata = sc.AnnData(X=np.asarray(emb, np.float32))
    adata.obs['cell_type'] = pd.Categorical(cell_types)
    adata.obs['batch'] = pd.Categorical(batches)
    adata.obs['row_idx'] = row_idx.astype(int)
    sc.pp.neighbors(adata, n_neighbors=int(cfg['n_neighbors']), use_rep='X')
    sc.tl.umap(adata, random_state=GLOBAL_SEED)
    sc.tl.leiden(
        adata, resolution=float(cfg['leiden_resolution']),
        key_added='leiden', random_state=GLOBAL_SEED,
    )
    leiden = adata.obs['leiden'].astype(str).values
    nmi = float(normalized_mutual_info_score(cell_types, leiden))
    ari = float(adjusted_rand_score(cell_types, leiden))
    raw_cell = safe_silhouette(emb, cell_types, cfg['silhouette_sample_size'])
    asw_cell = float(np.clip((raw_cell + 1.0) / 2.0, 0.0, 1.0))
    asw_batch, graph_conn, detail = conditioned_batch_metrics(emb, cell_types, batches, cfg)
    avg_bio = float(np.nanmean([nmi, np.clip(ari, 0.0, 1.0), asw_cell]))
    avg_batch = float(np.nanmean([asw_batch, graph_conn]))
    total_weight = float(cfg['bio_weight']) + float(cfg['batch_weight'])
    if total_weight <= 0:
        raise ValueError('bio_weight + batch_weight 必须大于0。')
    overall = float(
        float(cfg['bio_weight']) / total_weight * avg_bio
        + float(cfg['batch_weight']) / total_weight * avg_batch
    )
    metrics = {
        'nmi': nmi, 'ari': ari, 'asw_cell_raw': raw_cell, 'asw_cell': asw_cell,
        'asw_batch': asw_batch, 'graph_conn': graph_conn,
        'avg_bio': avg_bio, 'avg_batch': avg_batch, 'overall_score': overall,
        'n_celltypes_for_batch_metrics': int(len(detail)),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    detail.to_csv(run_dir / 'celltype_batch_metrics.csv', index=False)
    for color in ['cell_type', 'batch']:
        sc.pl.umap(adata, color=color, show=False, frameon=False)
        plt.savefig(run_dir / f'umap_{color}.png', dpi=300, bbox_inches='tight')
        plt.close()
    adata.uns['integration_metrics'] = metrics
    adata.write_h5ad(run_dir / 'zeroshot_encoder_eval_embedding.h5ad')
    return metrics


## 4. Benchmark 主循环


In [10]:
def safe_name(name: str) -> str:
    return ''.join('_' if ch in '<>:"/\\|?*' else ch for ch in str(name))


def checkpoint_signature(path: str) -> Dict[str, Any]:
    checkpoint = Path(path)
    stat = checkpoint.stat()
    return {
        'path': str(checkpoint.resolve()), 'size': int(stat.st_size), 'mtime_ns': int(stat.st_mtime_ns)
    }


def task_signature(model_cfg: Dict[str, Any], cfg: Dict[str, Any]) -> str:
    payload = {
        'pipeline_version': cfg['pipeline_version'],
        'dataset': cfg['dataset_name'],
        'checkpoint': checkpoint_signature(model_cfg['checkpoint_path']),
        'evaluation_split': cfg['evaluation_split'],
        'l2_normalize': cfg['l2_normalize_for_evaluation'],
        'native_preprocessing': {
            'input_values_are_binned': cfg['input_values_are_binned'],
            'num_bins': cfg['num_bins'],
            'keep_first_n_tokens': cfg['keep_first_n_tokens'],
            'cls_expr_value': cfg['cls_expr_value'],
            'pad_expr_value': cfg['pad_expr_value'],
        },
        'metrics': {key: cfg[key] for key in [
            'n_neighbors', 'leiden_resolution', 'min_cells_per_celltype',
            'min_batches_per_celltype', 'silhouette_sample_size', 'bio_weight', 'batch_weight'
        ]},
    }
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()[:16]


def run_one_model(model_cfg: Dict[str, Any], context: Any, cfg: Dict[str, Any]) -> Dict[str, Any]:
    model_name = str(model_cfg['model_name'])
    run_dir = OUTPUT_ROOT / safe_name(model_name)
    run_dir.mkdir(parents=True, exist_ok=True)
    signature = task_signature(model_cfg, cfg)
    metrics_path = run_dir / 'metrics.json'
    if cfg['skip_finished'] and metrics_path.exists():
        old = json.loads(metrics_path.read_text(encoding='utf-8'))
        if old.get('status') == 'finished' and old.get('task_signature') == signature:
            print(f'SKIP validated result: {model_name} | {signature}')
            return old

    model: Optional[nn.Module] = None
    try:
        print(f'START: {model_name} | zeroshot_encoder | {signature}')
        input_audit = audit_encoder_input(context.embed_loader_all, cfg)
        (run_dir / 'input_audit.json').write_text(
            json.dumps(input_audit, ensure_ascii=False, indent=2), encoding='utf-8'
        )
        model = build_model_shell(model_cfg)
        checkpoint_info = load_encoder_side_weights(model, model_cfg['checkpoint_path'])
        model.to(cfg['device'])
        raw_emb, row_idx = encode_all_cells(
            model, context.embed_loader_all, cfg['device'], cfg, f'Encode {model_name}'
        )
        normalized = l2_normalize(raw_emb) if cfg['l2_normalize_for_evaluation'] else raw_emb
        np.save(run_dir / 'encoder_cell_embeddings_raw.npy', raw_emb.astype(np.float32))
        np.save(run_dir / 'encoder_cell_embeddings.npy', normalized.astype(np.float32))
        np.save(run_dir / 'row_idx.npy', row_idx.astype(np.int64))
        metric_emb, metric_rows = select_evaluation_rows(normalized, row_idx, context, cfg)
        metrics = evaluate_embedding(metric_emb, metric_rows, context, run_dir, cfg)
        result = {
            **metrics, **checkpoint_info, 'input_audit': input_audit, 'status': 'finished',
            'dataset': cfg['dataset_name'], 'model_name': model_name,
            'run_mode': 'zeroshot_encoder', 'task_signature': signature,
            'embedding_dim': int(metric_emb.shape[1]),
            'n_cells': int(len(metric_emb)), 'n_cells_all': int(len(raw_emb)),
            'evaluation_split': cfg['evaluation_split'],
        }
    except Exception as e:
        traceback.print_exc()
        result = {
            'status': 'failed', 'error': repr(e), 'traceback': traceback.format_exc(),
            'dataset': cfg['dataset_name'], 'model_name': model_name,
            'run_mode': 'zeroshot_encoder', 'task_signature': signature,
        }
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    metrics_path.write_text(
        json.dumps(result, ensure_ascii=False, indent=2, default=str), encoding='utf-8'
    )
    print(f"DONE: {model_name} | {result['status']}")
    return result


def run_benchmark(models: List[Dict[str, Any]], context: Any, cfg: Dict[str, Any]) -> pd.DataFrame:
    results = [run_one_model(dict(model_cfg), context, cfg) for model_cfg in models]
    summary = pd.DataFrame(results)
    summary.to_csv(OUTPUT_ROOT / 'summary_metrics.csv', index=False)
    return summary


## 5. 运行与排名


In [11]:
summary_df = run_benchmark(USER_CONFIG['models'], context, CFG)
summary_df


START: geosketch_50_encoderonly | zeroshot_encoder | ea4fcada53f0e359
native preprocessing audit: {"input_values_are_binned": false, "num_bins": 51, "raw_min": 1.0, "raw_max": 1471.0, "raw_non_integer_ratio": 0.0, "binned_min": 1.0, "binned_max": 50.0, "binned_unique_preview": [1.0, 6.0, 17.0, 19.0, 24.0, 27.0, 30.0, 32.0, 33.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0], "valid_length_min": 316, "valid_length_max": 2049, "cls_value_matches": true}
checkpoint state source: model_state_dict
encoder modules: ('shared_token_embedding', 'token_norm', 'value_encoder', 'mask_flag_embedding', 'encoder')
[OK] loaded encoder tensors=156; ignored non-encoder tensors=4
ignored examples: ['value_head.net.0.weight', 'value_head.net.0.bias', 'value_head.net.3.weight', 'value_head.net.3.bias']


Encode geosketch_50_encoderonly: 100%|██████████| 5396/5396 [09:17<00:00,  9.68it/s]


DONE: geosketch_50_encoderonly | finished


,nmi,ari,asw_cell_raw,asw_cell,asw_batch,graph_conn,avg_bio,avg_batch,overall_score,n_celltypes_for_batch_metrics,...,input_audit,status,dataset,model_name,run_mode,task_signature,embedding_dim,n_cells,n_cells_all,evaluation_split
0,0.208589,0.074111,-0.242419,0.378791,0.747469,0.597868,0.220497,0.672669,0.401366,34,...,"{'input_values_are_binned': False, 'num_bins':...",finished,Renal,geosketch_50_encoderonly,zeroshot_encoder,ea4fcada53f0e359,512,97125,97125,all


In [12]:
batch = next(iter(context.embed_loader_all))

raw_values = batch["expressions"]
binned_values = prepare_encoder_values(batch, CFG)
padding_mask = batch["padding_mask"]

position = torch.arange(raw_values.shape[1])[None, :]
valid = (~padding_mask) & position.gt(0)

raw_x = raw_values[valid]
binned_x = binned_values[valid]

print("raw min/max:", raw_x.min().item(), raw_x.max().item())
print("binned min/max:", binned_x.min().item(), binned_x.max().item())
print("binned unique:", torch.unique(binned_x))

raw min/max: 1.0 4348.0
binned min/max: 1.0 50.0
binned unique: tensor([ 1., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23.,
        24., 25., 26., 27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37.,
        38., 39., 40., 41., 42., 43., 44., 45., 46., 47., 48., 49., 50.])


In [ ]:
if 'status' in summary_df.columns:
    ranked_df = summary_df.loc[summary_df['status'].astype(str).eq('finished')].copy()
else:
    ranked_df = pd.DataFrame()
if not ranked_df.empty and 'overall_score' in ranked_df.columns:
    ranked_df['overall_rank'] = ranked_df['overall_score'].rank(ascending=False, method='min')
    ranked_df = ranked_df.sort_values(
        ['overall_score', 'avg_bio', 'avg_batch'], ascending=False
    ).reset_index(drop=True)
ranked_df.to_csv(OUTPUT_ROOT / 'ranked_metrics.csv', index=False)
ranked_df
